# Slop Minimization Pipeline (Colab)

Trains **only** the three critical components, then saves them:

1. **Slop classifier** (DistilBERT, curriculum) → `outputs/classifier_curriculum`
2. **Prompt optimization** (TinyLlama as frozen generator + classifier as reward) → `outputs/prompt_opt/run_*`
3. **Slop rewriter** (T5 on rule-based slop pairs) → `outputs/slop_rewriter`

**TinyLlama** is the generator for prompt optimization (loaded from HuggingFace, not trained here).

In [ ]:
# Setup: clone into a fixed path. Cd to /content first so we never sit in a dir we're about to delete.
# Run this cell first. PROJECT_ROOT is set so other cells know where to run.
import os
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/ian-lent/slop-minimization.git"
CLONE_DIR = "/content/slop-repo"

# Ensure kernel cwd is valid (avoid getcwd errors after rm -rf of current dir)
get_ipython().run_line_magic('cd', '/content')
!rm -rf /content/slop-repo
!git clone $REPO_URL /content/slop-repo

# Repo is flat: configs, scripts, src are at clone root (no inner slop-minimization/)
PROJECT_ROOT = "/content/slop-repo"
print("Project root:", PROJECT_ROOT)
!ls {PROJECT_ROOT}/configs {PROJECT_ROOT}/scripts {PROJECT_ROOT}/src

In [ ]:
!pip -q install --upgrade pip
!pip -q install torch torchvision torchaudio
!pip -q install transformers datasets peft accelerate pyyaml tqdm scikit-learn sentencepiece

## 1. Build data

In [ ]:
# Run in project root (same shell so cwd is correct)
get_ipython().system(f'cd {PROJECT_ROOT} && python scripts/build_data.py --output-dir data')
get_ipython().system(f'ls {PROJECT_ROOT}/data')
get_ipython().system(f'wc -l {PROJECT_ROOT}/data/train.jsonl {PROJECT_ROOT}/data/val.jsonl {PROJECT_ROOT}/data/test.jsonl')

## 2. Train slop classifier (curriculum)

In [ ]:
# Use absolute PYTHONPATH and cd in the same shell (Colab ! does not inherit kernel cwd)
import os
cmd = f'cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}/src python scripts/train_token_classifier.py --config configs/classifier_encoder.yaml --output-dir outputs/classifier_curriculum'
get_ipython().system(cmd)
get_ipython().system(f'ls -la {PROJECT_ROOT}/outputs/classifier_curriculum/')

## 3. Slop rewriter: generate pairs + train T5

In [ ]:
get_ipython().system(f'cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}/src python scripts/train_slop_generator.py generate --input data/train.jsonl --output data/slop_pairs.jsonl')
get_ipython().system(f'wc -l {PROJECT_ROOT}/data/slop_pairs.jsonl')

In [ ]:
get_ipython().system(f'cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}/src python scripts/train_slop_generator.py train --train-path data/slop_pairs.jsonl --output-dir outputs/slop_rewriter --model-name t5-small --epochs 3')
get_ipython().system(f'ls -la {PROJECT_ROOT}/outputs/slop_rewriter/')

## 4. Prompt optimization (TinyLlama + classifier)

In [ ]:
get_ipython().system(f'cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}/src python scripts/optimize_prompts.py --config configs/prompt_opt.yaml')
get_ipython().system(f'ls -la {PROJECT_ROOT}/outputs/prompt_opt/')

## 5. Save all artifacts

In [ ]:
# Zip classifier, slop_rewriter, and prompt_opt for download
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_ROOT)
artifacts = [
    "outputs/classifier_curriculum",
    "outputs/slop_rewriter",
    "outputs/prompt_opt",
]
for d in artifacts:
    p = Path(d)
    if p.exists():
        print(f"Including: {d}")
    else:
        print(f"Missing: {d}")

zip_path = Path("slop_pipeline_artifacts.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", "outputs")
print(f"\nSaved: {zip_path.resolve()}")
print("Download this zip from the Colab file browser (or use Files pane).")

In [ ]:
# Optional: copy to Google Drive (run only if Drive is mounted)
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_ROOT)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

DRIVE_BASE = "/content/drive/MyDrive/slop_pipeline"
Path(DRIVE_BASE).mkdir(parents=True, exist_ok=True)

for name, src in [
    ("classifier_curriculum", "outputs/classifier_curriculum"),
    ("slop_rewriter", "outputs/slop_rewriter"),
    ("prompt_opt", "outputs/prompt_opt"),
]:
    src_path = Path(src)
    if src_path.exists():
        dst = Path(DRIVE_BASE) / name
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_path, dst, dirs_exist_ok=True)
        print(f"Copied {name} -> {dst}")
print(f"\nAll artifacts saved under {DRIVE_BASE}")